# 🛡️ SecurAI — Entraînement d'un Autoencoder de Dénoising

Ce notebook entraîne un petit CNN Autoencoder qui apprend à **reconstruire une image faciale propre** depuis une version adversariale (FGSM).  
Le modèle exporté (`denoiser.pt`) remplacera le pipeline NL-Means dans `modules/defender.py`.

**Pipeline :**
```
image_adv → [Encoder CNN] → latent (128D) → [Decoder CNN] → image_clean
```

**Loss = MSE(pixel) + λ · MSE(embedding FaceNet)**  
La perte perceptuelle préserve l'identité faciale.

In [ ]:
# 1️⃣ Monter Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2️⃣ Installer les dépendances
!pip uninstall -y -q jax jaxlib opencv-python opencv-contrib-python
!pip install -q "numpy==1.26.4" facenet-pytorch==2.6.0 torch torchvision tqdm opencv-python-headless scikit-learn joblib

import numpy as np
print(f"numpy {np.__version__}")
if np.__version__.startswith('2.'):
    import os; os.kill(os.getpid(), 9)

In [ ]:
# 3️⃣ Décompresser le projet
import os, zipfile, pathlib

zip_path = '/content/drive/MyDrive/Vision_Project.zip'
dest_dir = '/content/Vision_Project'

if not os.path.isdir(dest_dir):
    os.makedirs(dest_dir, exist_ok=True)
    print(f"Décompression de {zip_path} → {dest_dir}...")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(dest_dir)
    print("Terminé.")
else:
    print("Déjà décompressé.")

In [ ]:
# 4️⃣ Détecter la racine du projet
import sys

def find_modules_root(start_path: pathlib.Path) -> pathlib.Path:
    for root, dirs, _ in os.walk(start_path):
        if 'modules' in dirs:
            return pathlib.Path(root)
    raise FileNotFoundError("Dossier 'modules' introuvable.")

PROJECT_ROOT = find_modules_root(pathlib.Path(dest_dir))
sys.path.append(str(PROJECT_ROOT))

DATASET_ROOT = PROJECT_ROOT / 'dataset'
CLEAN_DIR    = DATASET_ROOT / 'clean'
ADV_DIR      = DATASET_ROOT / 'adv'
MODEL_DIR    = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

n_clean = len(list(CLEAN_DIR.glob('*.jpg'))) + len(list(CLEAN_DIR.glob('*.png')))
n_adv   = len(list(ADV_DIR.glob('*.jpg')))   + len(list(ADV_DIR.glob('*.png')))
print(f"Projet : {PROJECT_ROOT}")
print(f"Images clean : {n_clean}   |   Images adv : {n_adv}")

In [ ]:
# 5️⃣ Charger FaceNet pour la loss perceptuelle
from modules.face_recognizer import FaceRecognizer
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

recognizer = FaceRecognizer(mode='standard')
facenet = recognizer.model.eval().to(device)

# On gèle FaceNet — il sert uniquement à calculer la loss perceptuelle
for p in facenet.parameters():
    p.requires_grad = False

print("FaceNet chargé et gelé.")

In [ ]:
# 6️⃣ Générer les paires (adv_simulée, clean) à partir des images clean
# On simule les attaques FGSM localement pour avoir des paires parfaitement alignées.

import cv2
import numpy as np
from pathlib import Path

IMG_SIZE = 112   # taille d'entrée du Autoencoder (même que FaceNet 112×112)

def simulate_fgsm(img_bgr: np.ndarray, eps_range=(4, 16)) -> np.ndarray:
    """Simule une attaque FGSM avec epsilon aléatoire dans eps_range."""
    eps = np.random.randint(*eps_range)
    noise = np.random.choice([-eps, eps], size=img_bgr.shape).astype(np.int16)
    adv = np.clip(img_bgr.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    return adv

clean_images, adv_images = [], []
exts = {'.jpg', '.jpeg', '.png'}

for f in sorted(CLEAN_DIR.iterdir()):
    if f.suffix.lower() not in exts:
        continue
    img = cv2.imread(str(f))
    if img is None:
        continue
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    adv = simulate_fgsm(img)
    clean_images.append(img)
    adv_images.append(adv)

# Ajouter aussi les vraies images adversariales du dossier adv/
for f in sorted(ADV_DIR.iterdir()):
    if f.suffix.lower() not in exts:
        continue
    adv = cv2.imread(str(f))
    if adv is None:
        continue
    adv = cv2.resize(adv, (IMG_SIZE, IMG_SIZE))
    # On crée la version «propre» en appliquant un léger lissage
    clean_approx = cv2.GaussianBlur(adv, (5, 5), 0)
    clean_images.append(clean_approx)
    adv_images.append(adv)

print(f"Paires générées : {len(clean_images)}")

In [ ]:
# 7️⃣ Créer le Dataset PyTorch
import torch
from torch.utils.data import Dataset, DataLoader, random_split

def to_tensor(img_bgr: np.ndarray) -> torch.Tensor:
    """BGR uint8 [0,255] → RGB float32 [0,1] (C,H,W)"""
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return torch.from_numpy(rgb).permute(2, 0, 1)

class DenoisingDataset(Dataset):
    def __init__(self, clean_list, adv_list):
        self.clean = clean_list
        self.adv   = adv_list

    def __len__(self):
        return len(self.clean)

    def __getitem__(self, idx):
        return to_tensor(self.adv[idx]), to_tensor(self.clean[idx])

full_ds   = DenoisingDataset(clean_images, adv_images)
train_len = int(0.85 * len(full_ds))
val_len   = len(full_ds) - train_len
train_ds, val_ds = random_split(full_ds, [train_len, val_len])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)

print(f"Train : {train_len} | Val : {val_len}")

In [ ]:
# 8️⃣ Définir l'Autoencoder CNN
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class DenoisingAutoencoder(nn.Module):
    """
    Autoencoder léger : 112×112×3 → 14×14×128 (latent) → 112×112×3
    Compatible TorchScript pour export dans defender.py.
    """
    def __init__(self):
        super().__init__()
        # Encoder
        self.enc1 = ConvBlock(3, 32)          # 112→112
        self.pool1 = nn.MaxPool2d(2)          # →56
        self.enc2 = ConvBlock(32, 64)         # 56→56
        self.pool2 = nn.MaxPool2d(2)          # →28
        self.enc3 = ConvBlock(64, 128)        # 28→28
        self.pool3 = nn.MaxPool2d(2)          # →14

        # Bottleneck
        self.bottleneck = ConvBlock(128, 128) # 14→14

        # Decoder
        self.up3   = nn.ConvTranspose2d(128, 64, 2, stride=2)  # 14→28
        self.dec3  = ConvBlock(128, 64)       # skip de enc3
        self.up2   = nn.ConvTranspose2d(64, 32, 2, stride=2)   # 28→56
        self.dec2  = ConvBlock(64, 32)        # skip de enc2
        self.up1   = nn.ConvTranspose2d(32, 16, 2, stride=2)   # 56→112
        self.dec1  = ConvBlock(32, 16)        # skip de enc1

        self.final = nn.Conv2d(16, 3, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b  = self.bottleneck(self.pool3(e3))

        # Decoder avec skip connections
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.sigmoid(self.final(d1))

model = DenoisingAutoencoder().to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Autoencoder prêt — {total_params:,} paramètres entraînables")

In [ ]:
# 9️⃣ Définir la loss combinée : MSE pixel + loss perceptuelle FaceNet
import torch.nn.functional as F

LAMBDA_PERC = 0.5   # pondération de la loss perceptuelle (ajustable)

def preprocess_for_facenet(batch_rgb_01: torch.Tensor) -> torch.Tensor:
    """Redimensionne et normalise pour FaceNet (160×160, mean/std ImageNet)."""
    resized = F.interpolate(batch_rgb_01, size=(160, 160), mode='bilinear', align_corners=False)
    mean = torch.tensor([0.5, 0.5, 0.5], device=resized.device).view(1, 3, 1, 1)
    std  = torch.tensor([0.5, 0.5, 0.5], device=resized.device).view(1, 3, 1, 1)
    return (resized - mean) / std

def combined_loss(output, target):
    # 1. MSE pixel
    mse = F.mse_loss(output, target)

    # 2. Loss perceptuelle sur embeddings FaceNet
    with torch.no_grad():
        emb_target = facenet(preprocess_for_facenet(target))
    emb_output = facenet(preprocess_for_facenet(output))
    perc = F.mse_loss(emb_output, emb_target)

    return mse + LAMBDA_PERC * perc

print("Loss définie : MSE_pixel + 0.5 × MSE_embedding_FaceNet")

In [ ]:
# 🔟 Entraîner l'Autoencoder
from tqdm.notebook import tqdm

EPOCHS    = 30
LR        = 1e-3
PATIENCE  = 7   # early stopping

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

best_val_loss = float('inf')
epochs_no_improve = 0
best_state = None

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    train_loss = 0.0
    for adv_batch, clean_batch in tqdm(train_loader, desc=f"Époque {epoch:02d}/{EPOCHS} [Train]", leave=False):
        adv_batch   = adv_batch.to(device)
        clean_batch = clean_batch.to(device)

        optimizer.zero_grad()
        output = model(adv_batch)
        loss   = combined_loss(output, clean_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # --- Validation ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for adv_batch, clean_batch in val_loader:
            adv_batch   = adv_batch.to(device)
            clean_batch = clean_batch.to(device)
            output      = model(adv_batch)
            val_loss   += combined_loss(output, clean_batch).item()
    val_loss /= len(val_loader)

    scheduler.step(val_loss)

    print(f"Époque {epoch:02d}/{EPOCHS}  |  Train: {train_loss:.4f}  |  Val: {val_loss:.4f}")

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping à l'époque {epoch}.")
            break

print(f"\nMeilleure val loss : {best_val_loss:.4f}")
model.load_state_dict(best_state)

In [ ]:
# 1️⃣1️⃣ Exporter en TorchScript et sauvegarder
import shutil

model.eval().cpu()
scripted = torch.jit.script(model)

denoiser_path = MODEL_DIR / 'denoiser.pt'
scripted.save(str(denoiser_path))
print(f"Denoiser sauvegardé : {denoiser_path}")

# Créer un ZIP pour téléchargement
zip_out = '/content/denoiser_model.zip'
shutil.make_archive(zip_out.replace('.zip', ''), 'zip', root_dir=str(MODEL_DIR), base_dir='.')
print(f"Archive prête : {zip_out}")

In [ ]:
# 1️⃣2️⃣ Visualiser quelques reconstructions
import matplotlib.pyplot as plt

model.eval().to(device)
samples_adv, samples_clean = next(iter(val_loader))
samples_adv   = samples_adv[:6].to(device)
samples_clean = samples_clean[:6]

with torch.no_grad():
    reconstructed = model(samples_adv).cpu()

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for i in range(6):
    # Adversariale
    axes[0, i].imshow(samples_adv[i].cpu().permute(1,2,0).numpy())
    axes[0, i].set_title('Adv', fontsize=8)
    axes[0, i].axis('off')
    # Reconstruite
    axes[1, i].imshow(reconstructed[i].permute(1,2,0).numpy())
    axes[1, i].set_title('Dénoised', fontsize=8)
    axes[1, i].axis('off')
    # Clean cible
    axes[2, i].imshow(samples_clean[i].permute(1,2,0).numpy())
    axes[2, i].set_title('Clean', fontsize=8)
    axes[2, i].axis('off')

plt.suptitle('Autoencoder — Adv vs Dénoised vs Clean', fontsize=12)
plt.tight_layout()
plt.savefig('/content/denoiser_preview.png', dpi=150)
plt.show()
print("Aperçu sauvegardé : /content/denoiser_preview.png")

## 🔌 Intégration dans `modules/defender.py`

Après avoir téléchargé `denoiser.pt` et placé dans `securai_store/models/`, ajoutez dans `Defender.__init__()` :

```python
self.denoiser_path = os.path.join(base_dir, 'models', 'denoiser.pt')
self.denoiser = None
if os.path.exists(self.denoiser_path):
    self.denoiser = torch.jit.load(self.denoiser_path, map_location=self.device)
    self.denoiser.eval()
    print(f"[Defender] Denoiser chargé depuis {self.denoiser_path}")
```

Et remplacez `clean_image()` par :

```python
def clean_image(self, img: np.ndarray, ref_clean=None) -> np.ndarray:
    if self.denoiser is not None:
        # Autoencoder path
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        t = torch.from_numpy(rgb).permute(2,0,1).unsqueeze(0).to(self.device)
        import torch.nn.functional as F
        t = F.interpolate(t, size=(112, 112), mode='bilinear', align_corners=False)
        with torch.no_grad():
            out = self.denoiser(t).squeeze(0).permute(1,2,0).cpu().numpy()
        out = (out * 255).clip(0, 255).astype(np.uint8)
        out = cv2.cvtColor(out, cv2.COLOR_RGB2BGR)
        out = cv2.resize(out, (img.shape[1], img.shape[0]))
        return out
    else:
        # Fallback Feature Squeezing + NL-Means
        return self._classic_clean(img, ref_clean)
```

In [ ]:
# 1️⃣3️⃣ Télécharger le modèle
from google.colab import files
files.download('/content/denoiser_model.zip')